# 02 | Exploratory Data Analysis

Compare historical sales over time, across genres and platforms, and among regions. Identify top publishers and inspect correlations among numeric fields. Every chart is saved in `outputs/figures/`.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'vgsales_clean.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
if not CLEAN_PATH.exists():
    raise FileNotFoundError('Run 01_data_cleaning.ipynb first')

sales = pd.read_csv(CLEAN_PATH)
sns.set_theme(style='whitegrid', context='notebook')
print(f'{len(sales):,} titles | {sales.Year.min()}–{sales.Year.max()}')
sales.head()

## Global sales by release year

Annual totals show how the volume of recorded sales changes across the dataset's release years.

In [ ]:
yearly_sales = sales.groupby('Year', as_index=False)['Global_Sales'].sum()
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=yearly_sales, x='Year', y='Global_Sales', marker='o', ax=ax)
ax.set(title='Global video game sales by release year', xlabel='Release year', ylabel='Global sales (millions)')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'sales_by_year.png', dpi=160, bbox_inches='tight')
plt.show()

## Genre performance

Summed global sales reveal which genres account for the largest historical volume.

In [ ]:
genre_sales = sales.groupby('Genre', as_index=False)['Global_Sales'].sum().sort_values('Global_Sales')
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=genre_sales, x='Global_Sales', y='Genre', color='#287c78', ax=ax)
ax.set(title='Global sales by genre', xlabel='Global sales (millions)', ylabel='')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'sales_by_genre.png', dpi=160, bbox_inches='tight')
plt.show()

interactive_genre = px.bar(
    genre_sales,
    x='Global_Sales',
    y='Genre',
    orientation='h',
    title='Interactive genre sales',
    labels={'Global_Sales': 'Global sales (millions)', 'Genre': 'Genre'},
    color='Global_Sales',
    color_continuous_scale='Viridis',
)
interactive_genre.write_html(FIGURE_DIR / 'genre_sales_interactive.html')
interactive_genre.show()

## Platform performance

The chart focuses on the 12 platforms with the highest cumulative global sales.

In [ ]:
platform_sales = (sales.groupby('Platform', as_index=False)['Global_Sales'].sum()
                 .nlargest(12, 'Global_Sales').sort_values('Global_Sales'))
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=platform_sales, x='Global_Sales', y='Platform', color='#d17b3f', ax=ax)
ax.set(title='Top platforms by global sales', xlabel='Global sales (millions)', ylabel='')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'sales_by_platform.png', dpi=160, bbox_inches='tight')
plt.show()

## Regional trends

Compare annual sales totals in North America, Europe, Japan, and other markets.

In [ ]:
regional_columns = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
regional_yearly = (sales.groupby('Year', as_index=False)[regional_columns].sum()
                   .melt(id_vars='Year', var_name='Region', value_name='Sales'))
regional_yearly['Region'] = regional_yearly['Region'].str.replace('_Sales', '', regex=False)
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=regional_yearly, x='Year', y='Sales', hue='Region', marker='o', ax=ax)
ax.set(title='Regional sales by release year', xlabel='Release year', ylabel='Sales (millions)')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'regional_sales_by_year.png', dpi=160, bbox_inches='tight')
plt.show()

## Publisher leaders

Publishers are ranked by total global sales; titles with missing publisher metadata remain grouped as `Unknown`.

In [ ]:
publisher_sales = (sales.groupby('Publisher', as_index=False)['Global_Sales'].sum()
                   .nlargest(10, 'Global_Sales').sort_values('Global_Sales'))
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=publisher_sales, x='Global_Sales', y='Publisher', color='#55719a', ax=ax)
ax.set(title='Top publishers by global sales', xlabel='Global sales (millions)', ylabel='')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'top_publishers.png', dpi=160, bbox_inches='tight')
plt.show()

## Numeric feature correlations

The correlation matrix includes release year, regional sales, and global sales. Correlation describes association, not causation.

In [ ]:
numeric_columns = ['Year', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
correlations = sales[numeric_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(correlations, annot=True, fmt='.2f', cmap='vlag', center=0, square=True, ax=ax)
ax.set_title('Correlation among year and sales features')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'feature_correlations.png', dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved figures to {FIGURE_DIR.relative_to(PROJECT_ROOT)}')